# Text4GCN with Synthetic Data

This notebook demonstrates how to use Text4GCN with synthetic data for testing and development purposes.


## 1. Setup and Imports


In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import text4gcn
from text4gcn.preprocess import TextPipeline
from text4gcn.builder import Frequency, Semantic, Syntactic
from text4gcn.models import GNN, Layer, Builder
from text4gcn.utils.device import get_device_manager, check_pytorch_installation

print(f"Text4GCN version: {text4gcn.__version__}")
print(f"Text4GCN author: {text4gcn.__author__}")


## 2. Generate Synthetic Data


In [ ]:
from synthetic_data_generator import SyntheticDataGenerator

# Generate synthetic data
generator = SyntheticDataGenerator("synthetic_data")
generator.generate_dataset("SyntheticR8", num_documents=1000, train_ratio=0.8)

# Check generated files
from pathlib import Path
data_path = Path("synthetic_data")
print(f"Generated files in {data_path.absolute()}:")
for file in data_path.glob("SyntheticR8.*"):
    print(f"  - {file.name} ({file.stat().st_size} bytes)")


## 3. Check PyTorch Installation and Device


In [ ]:
# Check PyTorch installation
pytorch_info = check_pytorch_installation()
print("PyTorch Installation Info:")
for key, value in pytorch_info.items():
    print(f"  {key}: {value}")

# Get device manager
device_manager = get_device_manager()
print(f"\nUsing device: {device_manager.device}")

# Get device information
device_info = device_manager.get_device_info()
print("\nDevice Information:")
for key, value in device_info.items():
    print(f"  {key}: {value}")


## 4. Text Preprocessing Pipeline


In [ ]:
# Create and execute TextPipeline
print("Creating TextPipeline...")
pipe = TextPipeline(
    dataset_name="SyntheticR8",
    rare_count=5,
    dataset_path="synthetic_data",
    language="english"
)

print("Executing TextPipeline...")
pipe.execute()

print("✅ TextPipeline completed successfully!")


## 5. Check Pipeline Output


In [ ]:
# Check output files
data_path = Path("synthetic_data")
dataset_name = "SyntheticR8"

print("📁 Pipeline Output Files:")

# Check cleaned data
cleaned_dir = data_path / f"{dataset_name}.cleaned"
if cleaned_dir.exists():
    print(f"  ✅ Cleaned data: {cleaned_dir}")
    cleaned_file = cleaned_dir / f"{dataset_name}.txt"
    if cleaned_file.exists():
        print(f"     - {cleaned_file.name} ({cleaned_file.stat().st_size} bytes)")

# Check shuffled data
shuffled_dir = data_path / f"{dataset_name}.shuffled"
if shuffled_dir.exists():
    print(f"  ✅ Shuffled data: {shuffled_dir}")
    for file in shuffled_dir.glob("*"):
        print(f"     - {file.name} ({file.stat().st_size} bytes)")

# Check node features
node_features_dir = data_path / f"{dataset_name}.node_features"
if node_features_dir.exists():
    print(f"  ✅ Node features: {node_features_dir}")
    for file in node_features_dir.glob("*"):
        print(f"     - {file.name} ({file.stat().st_size} bytes)")

# Check log files
log_dir = data_path / "log"
if log_dir.exists():
    print(f"  📝 Log files: {log_dir}")
    for file in log_dir.glob("*.txt"):
        print(f"     - {file.name} ({file.stat().st_size} bytes)")


## 6. Build Adjacency Matrix


In [ ]:
# Test frequency-based adjacency
print("Building frequency-based adjacency matrix...")
freq_builder = Frequency()
adjacency = freq_builder.tfidf(
    dataset_name="SyntheticR8",
    dataset_path="synthetic_data"
)
print(f"✅ Frequency adjacency matrix shape: {adjacency.shape}")

# Test semantic-based adjacency
print("\nBuilding semantic-based adjacency matrix...")
sem_builder = Semantic()
try:
    semantic_adj = sem_builder.word2vec(
        dataset_name="SyntheticR8",
        dataset_path="synthetic_data"
    )
    print(f"✅ Semantic adjacency matrix shape: {semantic_adj.shape}")
except Exception as e:
    print(f"⚠️  Semantic adjacency failed: {e}")
    print("   This is expected if Word2Vec dependencies are not available")


## 7. Train GNN Model


In [ ]:
# Create and train GNN model
print("Creating GNN model...")
model = GNN(
    dataset="SyntheticR8",
    path="synthetic_data",
    log_dir="logs",
    builder=Builder.Tfidf,
    layer=Layer.GCN,
    epochs=50,  # Reduced for demo
    dropout=0.5,
    device=device_manager.device  # Use detected device
)

print("Training GNN model...")
try:
    model.fit()
    print("✅ GNN model training completed!")
except Exception as e:
    print(f"⚠️  GNN training failed: {e}")
    print("   This might be due to missing dependencies or data issues")


## 8. Converting Documents to Text4GCN Format

This section demonstrates how to convert a list of documents into the format required by Text4GCN for node classification.


In [ ]:
# Example: Converting a list of documents to Text4GCN format

def convert_documents_to_text4gcn_format(documents, labels, train_ratio=0.8, output_dir="custom_data"):
    """
    Convert a list of documents to Text4GCN format.
    
    Args:
        documents: List of text documents
        labels: List of corresponding labels
        train_ratio: Ratio of documents for training (default: 0.8)
        output_dir: Output directory for files
    
    Returns:
        str: Path to the created dataset directory
    """
    import os
    import random
    from pathlib import Path
    
    # Create output directory
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    # Validate inputs
    if len(documents) != len(labels):
        raise ValueError("Number of documents must match number of labels")
    
    # Shuffle data
    data = list(zip(documents, labels))
    random.shuffle(data)
    documents, labels = zip(*data)
    
    # Split into train/test
    num_train = int(len(documents) * train_ratio)
    train_docs = documents[:num_train]
    train_labels = labels[:num_train]
    test_docs = documents[num_train:]
    test_labels = labels[num_train:]
    
    # Create .txt file (one document per line)
    txt_file = output_path / "custom_dataset.txt"
    with open(txt_file, 'w', encoding='utf-8') as f:
        for doc in documents:
            f.write(doc + '\n')
    
    # Create .meta file (id, split, label)
    meta_file = output_path / "custom_dataset.meta"
    with open(meta_file, 'w', encoding='utf-8') as f:
        for i, (doc, label) in enumerate(zip(documents, labels)):
            split = "train" if i < num_train else "test"
            f.write(f"{i}\t{split}\t{label}\n")
    
    print(f"✅ Created Text4GCN format files:")
    print(f"   - {txt_file} ({txt_file.stat().st_size} bytes)")
    print(f"   - {meta_file} ({meta_file.stat().st_size} bytes)")
    print(f"   - Total documents: {len(documents)}")
    print(f"   - Train documents: {num_train}")
    print(f"   - Test documents: {len(documents) - num_train}")
    print(f"   - Unique labels: {len(set(labels))}")
    
    return str(output_path)

# Example usage with sample documents
sample_documents = [
    "This is a document about machine learning and artificial intelligence algorithms.",
    "The company reported strong quarterly earnings and revenue growth.",
    "The research study shows promising results in medical treatment.",
    "The sports team won the championship with excellent performance.",
    "Technology innovation drives digital transformation in businesses.",
    "Scientific research requires rigorous methodology and analysis.",
    "Business strategy focuses on market expansion and customer satisfaction.",
    "Athletic training improves physical performance and endurance.",
    "Healthcare professionals provide medical care and treatment.",
    "Computer programming involves coding and software development."
]

sample_labels = [
    "technology", "business", "science", "sports", "technology",
    "science", "business", "sports", "health", "technology"
]

# Convert to Text4GCN format
dataset_path = convert_documents_to_text4gcn_format(
    sample_documents, 
    sample_labels, 
    train_ratio=0.8,
    output_dir="custom_data"
)


### **Formato dos Arquivos**

O Text4GCN espera dois arquivos principais:

1. **`{dataset_name}.txt`** - Documentos de texto (um por linha)
2. **`{dataset_name}.meta`** - Metadados (id, split, label)

#### **Exemplo de arquivo .txt:**
```
This is a document about machine learning and artificial intelligence algorithms.
The company reported strong quarterly earnings and revenue growth.
The research study shows promising results in medical treatment.
```

#### **Exemplo de arquivo .meta:**
```
0	train	technology
1	train	business
2	test	science
```


In [ ]:
# Testar o dataset customizado com TextPipeline
print("🧪 Testando dataset customizado com TextPipeline...")

try:
    # Criar pipeline para o dataset customizado
    custom_pipe = TextPipeline(
        dataset_name="custom_dataset",
        rare_count=3,  # Menor para dataset pequeno
        dataset_path="custom_data",
        language="english"
    )
    
    # Executar pipeline
    custom_pipe.execute()
    print("✅ TextPipeline executado com sucesso no dataset customizado!")
    
    # Verificar arquivos de saída
    custom_data_path = Path("custom_data")
    print("\n📁 Arquivos gerados:")
    for item in custom_data_path.rglob("*"):
        if item.is_file():
            print(f"   - {item.relative_to(custom_data_path)} ({item.stat().st_size} bytes)")
            
except Exception as e:
    print(f"⚠️  Erro ao processar dataset customizado: {e}")
    print("   Isso pode ser normal para datasets muito pequenos")


## 9. Summary and Next Steps


In [ ]:
print("🎉 Text4GCN Synthetic Data Demo Complete!")
print("=" * 50)

print("\n📊 What we accomplished:")
print("  ✅ Generated synthetic data in Text4GCN format")
print("  ✅ Tested TextPipeline preprocessing")
print("  ✅ Built adjacency matrices")
print("  ✅ Trained GNN model")
print("  ✅ Demonstrated document conversion")

print("\n🔧 Next steps:")
print("  - Experiment with different synthetic data sizes")
print("  - Try different adjacency matrix builders")
print("  - Test with real benchmark datasets (R8, R52, etc.)")
print("  - Optimize model hyperparameters")
print("  - Convert your own documents to Text4GCN format")

print("\n📁 Generated files:")
data_path = Path("synthetic_data")
if data_path.exists():
    total_size = sum(f.stat().st_size for f in data_path.rglob('*') if f.is_file())
    print(f"  - Synthetic data: {total_size / 1024 / 1024:.2f} MB")
    print(f"  - Files: {len(list(data_path.rglob('*')))} items")

custom_path = Path("custom_data")
if custom_path.exists():
    total_size = sum(f.stat().st_size for f in custom_path.rglob('*') if f.is_file())
    print(f"  - Custom data: {total_size / 1024 / 1024:.2f} MB")
    print(f"  - Files: {len(list(custom_path.rglob('*')))} items")

print("\n💡 Tips for using your own data:")
print("  1. Prepare your documents as a list of strings")
print("  2. Create corresponding labels")
print("  3. Use convert_documents_to_text4gcn_format()")
print("  4. Run TextPipeline on the converted data")
print("  5. Build adjacency matrices and train models")

print("\n🚀 Ready to use Text4GCN with your own data!")
